In [15]:
import datetime
import os

import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl

from src.thetadata_pipeline.loaders.tick import ensure_stock_tick_windows, StockTickWindow
from src.thetadata_pipeline.pipeline_config import load_pipeline_config
from src.thetadata_pipeline.settings import get_settings
from src.thetadata_pipeline.time_utils import to_date

cfg = load_pipeline_config()
settings = get_settings()

DATE_CUT = datetime.date(2021, 1, 1)
TICKER: str = "SPY"
NEED_TIME: float = 9.6
CUT_DELTA_UP: float = 0.40
CUT_DELTA_DONW: float = 0.10
NEED_DELTA: float = 0.35

# Trades

In [ ]:
need_time = int(NEED_TIME * 1_000 * 60 * 60)
options_df = pl.DataFrame()
for file in settings.m1_with_greeks_dir.glob(f'*{TICKER}_m1_greeks_opts.parquet'):
    cur_df = (
        pl.read_parquet(file).filter(
            (pl.col('ms_of_day') >= need_time) &
            (
                ((pl.col('delta_bid') <= CUT_DELTA_UP) & (pl.col('delta_bid') >= CUT_DELTA_DONW) & (pl.col('right') == 'c')) |
                ((pl.col('delta_bid') >= -CUT_DELTA_UP) & (pl.col('delta_bid') <= -CUT_DELTA_DONW) & (pl.col('right') == 'p'))
            )
        )
    )
    options_df = pl.concat([options_df, cur_df], how='diagonal')

len(options_df['date'].unique())

In [ ]:
options_df = options_df.filter(pl.col('date') > DATE_CUT)

In [ ]:
options_df = options_df.with_columns(
        spread=(pl.col('ask') - pl.col('bid')) / pl.col('ask')
    ).filter(pl.col('spread') < 0.1)

start_df = options_df\
    .sort('date', 'ms_of_day')\
    .unique(['date', 'ms_of_day', 'right'], keep='first', maintain_order=True)\
    .group_by(['date', 'ms_of_day']).agg(pl.col('ticker').count())\
    .sort('date', 'ms_of_day')\
    .filter(pl.col('ticker') > 1)\
    .unique('date', keep='first', maintain_order=True)\
    .rename({'ms_of_day': 'start_time'})\
    .drop('ticker')
print(start_df.shape)

options_df = options_df.join(
        start_df,
        on='date',
        how='inner'
    ).filter(pl.col('ms_of_day') >= pl.col('start_time'))\
    .drop('spread', 'start_time')

In [ ]:
call_trades = options_df.filter(pl.col('right') == 'c')\
    .with_columns(delta_diff=(pl.col('delta_bid') - NEED_DELTA).abs())\
    .sort(['date', 'expiration', 'ms_of_day', 'delta_diff'])\
    .unique(subset=['date', 'expiration'], keep='first', maintain_order=True)\
    .drop([
        'bid_size', 'bid_exchange', 'bid_condition', 'ask_size', 'ask_exchange', 'ask_condition',
        'dgs1', 'timeToExp', 'gamma_ask', 'theta_ask', 'vega_ask', 'rho_ask', 'IV_ask', 'delta_ask',
        'gamma_bid', 'theta_bid', 'vega_bid', 'rho_bid', 'delta_diff'
    ])

put_trades = options_df.filter(pl.col('right') == 'p')\
    .with_columns(delta_diff=(pl.col('delta_bid') + NEED_DELTA).abs())\
    .sort(['date', 'expiration', 'ms_of_day', 'delta_diff'])\
    .unique(subset=['date', 'expiration'], keep='first', maintain_order=True)\
    .drop([
        'bid_size', 'bid_exchange', 'bid_condition', 'ask_size', 'ask_exchange', 'ask_condition',
        'dgs1', 'timeToExp', 'gamma_ask', 'theta_ask', 'vega_ask', 'rho_ask', 'IV_ask', 'delta_ask',
        'gamma_bid', 'theta_bid', 'vega_bid', 'rho_bid', 'delta_diff'
    ])

len(call_trades), len(put_trades)

In [ ]:
cur_df = call_trades[['date', 'IV_bid']].join(
        put_trades[['date', 'IV_bid']].rename({'IV_bid': 'IV_bid_put'}),
        on='date'
    ).with_columns(avgIV=(pl.col('IV_bid') + pl.col('IV_bid_put')) / 2)\
    [['date', 'avgIV']].to_pandas()\
    .set_index('date')\
    .rolling(window='365D')\
    .mean()

px.line(cur_df, title='Вся динамика капитала зависит от волатильности. Чем она выше, тем выше профит. Раньше эта стратегия была не доступна из-за спредов и малого количество экпираций в неделе.')

# Preparing for getting trade data

In [ ]:
ticker_m1 = pl.DataFrame()
for file in settings.stock_m1_dir.glob(f'*{TICKER}_m1_stock.parquet'):
    ticker_m1 = pl.concat([ticker_m1, pl.read_parquet(file)])
ticker_m1 = (
    ticker_m1.with_columns(pl.col('date').cast(pl.String).str.to_datetime('%Y%m%d').cast(pl.Date))
    .filter(pl.col('date') > DATE_CUT)
    .sort('date', 'ms_of_day')
)

In [98]:
def update_tick_data(df, shift: int = 300_000):
    tick_concurrency = 8
    windows = [
        StockTickWindow(
            ticker=TICKER,
            day=to_date(row["date_exit"]),
            start_ms=int(row["ms_of_day"]),
            end_ms=int(row["ms_of_day"]) + shift,
        )
        for row in df.iter_rows(named=True)
    ]

    tick_pool = ensure_stock_tick_windows(
        settings=settings,
        windows=windows,
        concurrency=tick_concurrency,
        interval="tick",
    )

    return tick_pool

# Call Trades

In [31]:
%%time
total_m1 = pl.DataFrame()
for row in call_trades.iter_rows(named=True):
    need_date = row['date']
    need_time = row['ms_of_day']
    
    cur_m1 = ticker_m1.filter(
            (need_date == pl.col('date')) & (need_time < pl.col('ms_of_day')) & 
            (row['strike'] < pl.col('high'))
        )
    if len(cur_m1) == 0:
        continue

    cur_m1 = cur_m1[0]['ms_of_day', 'high', 'date'].rename({'date': 'date_exit'})\
        .with_columns(
            expiration=pl.lit(row['expiration']),
            strike=pl.lit(row['strike']),
            date=pl.lit(row['date']),
            right=pl.lit(row['right']),
            ticker=pl.lit(row['ticker']),
        )
    total_m1 = pl.concat([total_m1, cur_m1])
total_m1 = total_m1.with_row_index(name='idx')

CPU times: total: 250 ms
Wall time: 1.02 s


In [ ]:
tick_df = update_tick_data(total_m1).rename({'time': 'quote_time', 'bid': 'quote_bid', 'ask': 'quote_ask'})

In [134]:
total_m1_with_ticks = (
    total_m1.join(tick_df, on='date', how='left').filter(pl.col('quote_ask') > pl.col('strike'))
    .sort('idx', 'date', 'quote_time')
    .unique('idx', keep="first", maintain_order=True)
)

idx_dif = set(total_m1['idx']) - set(total_m1_with_ticks['idx'])
empty_data = total_m1.filter(pl.col('idx').is_in(idx_dif))

empty_data = (
    empty_data.join(tick_df, on='date', how='left')
    .with_columns(diff=(pl.col('strike') - pl.col('quote_bid')).abs())
    .sort('idx', 'diff')
    .unique('idx', keep="first", maintain_order=True)
    .drop('diff')
)
total_m1 = pl.concat([total_m1_with_ticks, empty_data]).sort('idx').drop('quote_bid', 'quote_ask')

In [154]:
from src.thetadata_pipeline.loaders.tick import ensure_option_quote_window

for row in total_m1.iter_rows(named=True):
    ensure_option_quote_window(settings, row['ticker'], row['date_exit'], row['expiration'], row['strike'], row['right'], start_ms=)
    break


ImportError: cannot import name 'ensure_option_quote_window' from 'src.thetadata_pipeline.loaders.tick' (H:\Market\ThetaData\src\thetadata_pipeline\loaders\tick.py)

In [156]:
total_m1

idx,ms_of_day,high,date_exit,expiration,strike,date,right,ticker,quote_time
u32,i64,f64,date,date,f64,date,str,str,time
0,35460000,372.03,2021-01-06,2021-01-06,372.0,2021-01-06,"""c""","""SPY""",09:51:02.405
1,35760000,379.225,2021-01-11,2021-01-11,379.0,2021-01-11,"""c""","""SPY""",09:56:14.472
2,46080000,380.01,2021-01-13,2021-01-13,380.0,2021-01-13,"""c""","""SPY""",12:48:42.691
3,47340000,379.02,2021-01-19,2021-01-19,379.0,2021-01-19,"""c""","""SPY""",13:09:22.223
4,36300000,382.19,2021-01-20,2021-01-20,382.0,2021-01-20,"""c""","""SPY""",10:05:16.797
…,…,…,…,…,…,…,…,…,…
740,35280000,739.14,2026-05-11,2026-05-11,739.0,2026-05-11,"""c""","""SPY""",09:48:44.006
741,54780000,737.15,2026-05-12,2026-05-12,737.0,2026-05-12,"""c""","""SPY""",15:13:21.654
742,37320000,738.1,2026-05-13,2026-05-13,738.0,2026-05-13,"""c""","""SPY""",10:22:44.820


In [141]:
lattency = pd.read_csv(r"H:\Market\ThetaData\data\ib\U1717377_lattency.csv")

In [152]:
import numpy as np
lattency[lattency['lable'] == 'stop']['lattency'].median()

np.float64(0.221)

# M1 Chart

In [105]:
m1_ticker = pl.DataFrame()
for file in settings.stock_m1_dir.glob(f"*{TICKER}_m1_stock.parquet"):
    m1_ticker = pl.concat([m1_ticker, pl.read_parquet(file)])

m1_ticker = m1_ticker.with_columns(
    (pl.col("ms_of_day") * 1_000_000)
    .cast(pl.Time)
    .dt.to_string("%H:%M:%S")
    .alias("formatted_time")
).filter(pl.col('close') > 0)

In [116]:
from src.thetadata_pipeline.time_utils import format_time_ms

ms_of_day = 45000000
day = 20251028

format_time_ms(ms_of_day)

'12:30:00.000000'

In [117]:
import plotly.graph_objects as go

cur_df = m1_ticker.filter(pl.col('date') == day)
fig = go.Figure(data=[
    go.Candlestick(
        x=cur_df["formatted_time"],
        open=cur_df["open"],
        high=cur_df["high"],
        low=cur_df["low"],
        close=cur_df["close"]
    )
])

fig.update_layout(
    title="Intraday OHLC",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False
)

fig

In [93]:
import pandas as pd
tick_df.filter(pl.col('date') == pd.to_datetime(day, format='%Y%m%d'))

date,quote_time,quote_bid,quote_ask
date,time,f64,f64
2022-06-15,14:41:00,379.22,379.24
2022-06-15,14:41:00.008,379.21,379.24
2022-06-15,14:41:00.012,379.19,379.24
2022-06-15,14:41:00.017,379.2,379.24
2022-06-15,14:41:00.028,379.19,379.23
…,…,…,…
2022-06-15,14:42:00.174,380.02,380.01
2022-06-15,14:42:00.174,380.03,380.01
2022-06-15,14:42:00.175,380.03,380.02
